# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShubhamSnSharma/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [30]:
# Imports
import duckdb
from getpass import getpass

# Authenticate with Hugging Face
token = getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

Enter your Hugging Face READ token: ··········


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

---
**Answer**

**Unit of analysis –** One row represents one content item for one client on one reporting date (one page-day observation).

**Time window –** The dataset spans **2025-01-27 to 2026-06-30**. For verification and feature construction in this notebook, the **March 2026** partition (`month = '2026-03'`) is used as the analysis window, following the assignment guidance to use a mid-panel month during development.

In [31]:
# Verify the overall reporting period covered by the dataset.

con.execute(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date
0,2025-01-27,2026-06-30


In [32]:
# Verify that the March 2026 partition exists and summarize
# the reporting period and number of observations in the analysis window.
con.execute(f"""
SELECT
    MIN(report_date) AS march_start,
    MAX(report_date) AS march_end,
    COUNT(*) AS total_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""").df()

,march_start,march_end,total_rows
0,2026-03-01,2026-03-31,9841378


### Verification Summary

The verification queries confirm that:

- The warehouse covers data from **2025-01-27** to **2026-06-30**.
- The notebook uses the **March 2026** partition for analysis, which spans **2026-03-01** to **2026-03-31** and contains **9,841,378** observations.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

---
**Answer**

| Bucket | Fields | Reason |
|--------|--------|--------|
| **Feature** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `ga4_engaged_sessions` | Historical search and engagement metrics that are available before making a refresh decision and can be used as model inputs. |
| **Label / Proxy** | Refresh priority (proxy target defined in a later assignment) | This is the outcome the model will eventually predict, so it must not be used as an input feature. |
| **Context** | `report_date`, `client_hash_id`, `content_hash_id`, `month` | Used to identify observations and define the analysis window. These fields provide context but are not used as predictive features. |
| **Excluded** | `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` | Operational availability flags used for filtering valid records. They are excluded as model features because they describe data availability rather than content performance. |

In [33]:
# Display representative records containing the fields
# referenced in the data contract for the March 2026 analysis window.

con.execute(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    month,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_engaged_sessions,
    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
LIMIT 10
""").df()

,report_date,client_hash_id,content_hash_id,month,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03,20,0,3.350000,<NA>,<NA>,True,False,True,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03,1,0,0.000000,<NA>,<NA>,True,False,True,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03,125,1,4.928000,<NA>,<NA>,True,False,True,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03,7,0,4.000000,<NA>,<NA>,True,False,True,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03,11,0,2.272727,<NA>,<NA>,True,False,True,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03,239,1,7.347280,<NA>,<NA>,True,False,True,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03,191,0,7.832461,<NA>,<NA>,True,False,True,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03,55,0,3.272727,<NA>,<NA>,True,False,True,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03,77,0,5.636364,<NA>,<NA>,True,False,True,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03,2,0,4.500000,<NA>,<NA>,True,False,True,<NA>


### Verification Summary

The query confirms that the fields referenced in the data contract are present in the March 2026 analysis window. These fields will be used for feature engineering, filtering, and validation in the subsequent steps of the notebook.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [34]:
# Verify that each page-day observation is unique in the
# March 2026 analysis window.

con.execute(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [35]:
# Measure missing values for the selected feature columns
# in the March 2026 analysis window.

con.execute(f"""
SELECT
    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS gsc_impressions_missing,
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS gsc_clicks_missing,
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS gsc_avg_position_missing,
    AVG(CASE WHEN ga4_pageviews IS NULL THEN 1.0 ELSE 0 END) AS ga4_pageviews_missing,
    AVG(CASE WHEN ga4_engaged_sessions IS NULL THEN 1.0 ELSE 0 END) AS ga4_engaged_sessions_missing
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions_missing,gsc_clicks_missing,gsc_avg_position_missing,ga4_pageviews_missing,ga4_engaged_sessions_missing
0,0.0,0.0,0.633074,0.30674,0.30674


In [36]:
# Verify whether missing GA4 metrics follow the
# documented GA4 availability flag.

con.execute(f"""
SELECT
    ga4_data_available,
    COUNT(*) AS rows,
    AVG(CASE WHEN ga4_pageviews IS NULL THEN 1.0 ELSE 0 END) AS missing_pageviews,
    AVG(CASE WHEN ga4_engaged_sessions IS NULL THEN 1.0 ELSE 0 END) AS missing_engaged_sessions
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY ga4_data_available
ORDER BY ga4_data_available
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,rows,missing_pageviews,missing_engaged_sessions
0,False,6408671,0.0,0.0
1,True,413966,0.0,0.0
2,<NA>,3018741,1.0,1.0


### Verification Summary

The verification queries confirm that:

- No duplicate `(report_date, client_hash_id, content_hash_id)` combinations were found in the March 2026 analysis window, confirming the expected page-day grain.
- `gsc_impressions` and `gsc_clicks` contain no missing values in the March 2026 analysis window.
- Missing values were observed in `gsc_avg_position` (~63.3%) and the selected GA4 metrics (~30.7%).
- In the March 2026 analysis window, missing GA4 metrics were observed only on rows where `ga4_data_available` is `NULL`, while rows with `ga4_data_available = TRUE` or `FALSE` showed no missing GA4 values.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

---
**Answer**

This dataset has several limitations that should be considered before feature engineering or model development:

- **Uneven client history:** Clients have different amounts of historical data, so a single global time window may not represent all clients equally. Comparisons across clients should account for differences in history length.
- **Incomplete GA4 coverage:** Some rows contain only GSC data because GA4 was not yet available for those clients. Missing or unavailable GA4 values should not be interpreted as zero engagement.
- **Window alignment:** Different tables and metrics may represent different time windows. Features and labels must be aligned so that no future information is introduced into the prediction task.
- **Decision support, not causation:** The warehouse records search and analytics behaviour but cannot explain *why* a page's performance changed. External factors such as algorithm updates, seasonality, or content changes are not captured directly.

In [37]:
# Summarize the reporting history across all clients.

con.execute(f"""
SELECT
    COUNT(DISTINCT client_hash_id) AS clients,
    MIN(first_date) AS earliest_history,
    MAX(first_date) AS latest_history,
    MIN(observations) AS min_observations,
    MAX(observations) AS max_observations
FROM (
    SELECT
        client_hash_id,
        MIN(report_date) AS first_date,
        COUNT(*) AS observations
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY client_hash_id
)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clients,earliest_history,latest_history,min_observations,max_observations
0,70,2025-01-27,2026-05-28,2619,8708971


### Verification Summary

The verification query confirms that the warehouse contains **70 clients** with different reporting start dates and substantially different numbers of observations (ranging from **2,619** to **870,897** rows per client). This supports the need to account for unequal client history when selecting analysis windows and building predictive models.

## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.